# Week 1 Tuesday — Telco Churn EDA

Data source: Kaggle `blastchar/telco-customer-churn`, uploaded to `s3://beant-mlops-portfolio-666258711441/raw/`.

In [7]:
import boto3
import pandas as pd

BUCKET = "beant-mlops-portfolio-666258711441"
KEY = "raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=BUCKET, Key=KEY)
df = pd.read_csv(obj["Body"])
df.shape

(7043, 21)

In [8]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Reload (or reuse df from Monday)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Only 11 nulls (new customers, tenure=0) - safe to impute as 0 rather than drop
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# customerID is a unique identifier, not a feature - drop it
df = df.drop(columns=["customerID"])

# Target: convert Yes/No to 1/0
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["Churn"])

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [c for c in X.columns if c not in numeric_cols]


# Sanity-Check the split:

In [9]:
X["SeniorCitizen"] = X["SeniorCitizen"].astype(str)  # it's 0/1 int but semantically categorical
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print(numeric_cols)
print(categorical_cols)

['tenure', 'MonthlyCharges', 'TotalCharges']
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


# Column Transformer

In [10]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
print(X_train_t.shape, X_test_t.shape)

(5634, 46) (1409, 46)


In [11]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

role = "arn:aws:iam::666258711441:role/service-role/AmazonSageMaker-ExecutionRole-20260722T143599"

processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.t3.medium",
    instance_count=1,
)

processor.run(
    code="preprocessing.py",
    inputs=[
        ProcessingInput(
            source="s3://beant-mlops-portfolio-666258711441/raw/",
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/output/train",
                          destination="s3://beant-mlops-portfolio-666258711441/processed/train"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/output/test",
                          destination="s3://beant-mlops-portfolio-666258711441/processed/test"),
    ],
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2026-07-23-15-56-17-033


...........

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:13                                                                                   │
│                                                                                                  │
│   10 │   instance_count=1,                                                                       │
│   11 )                                                                                           │
│   12                                                                                             │
│ ❱ 13 processor.run(                                                                              │
│   14 │   code="preprocessing.py",                                                                │
│   15 │   inputs=[                                                                                │
│   16 │   │   ProcessingInput(                                                                    │
│                                                                                                  │
│ c:\dev\MLOPS_Training\mlops-env\Lib\site-packages\sagemaker\workflow\pipeline_context.py:346 in  │
│ wrapper                                                                                          │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ c:\dev\MLOPS_Training\mlops-env\Lib\site-packages\sagemaker\processing.py:697 in run             │
│                                                                                                  │
│    694 │   │   )                                                                                 │
│    695 │   │   self.jobs.append(self.latest_job)                                                 │
│    696 │   │   if wait:                                                                          │
│ ❱  697 │   │   │   self.latest_job.wait(logs=logs)                                               │
│    698 │                                                                                         │
│    699 │   def _include_code_in_inputs(self, inputs, code, kms_key=None):                        │
│    700 │   │   """Converts code to appropriate input and includes in input list.                 │
│                                                                                                  │
│ c:\dev\MLOPS_Training\mlops-env\Lib\site-packages\sagemaker\processing.py:1122 in wait           │
│                                                                                                  │
│   1119 │   │                                                                                     │
│   1120 │   │   """                                                                               │
│   1121 │   │   if logs:                                                                          │
│ ❱ 1122 │   │   │   self.sagemaker_session.logs_for_processing_job(self.job_name, wait=True)      │
│   1123 │   │   else:                                                                             │
│   1124 │   │   │   self.sagemaker_session.wait_for_processi